# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

A model score is not a product. This notebook turns the Week 4–6 work (hand rule → learned rankers → honest validation) into a **content action playbook**: a ranked review queue with reason codes, an archetype→action map, the decay/refresh finding, intended use and limits, human-review rules, a no-go list, cost/value framing, and light monitoring triggers. It exports the queue and figures that next week's paper builds on.

**Design decisions this notebook inherits from Week 6 (`work/outputs/validation_audit_metrics.json`), not re-argues:**

| Evidence (W6 receipts) | Consequence for the playbook |
|---|---|
| Client-grouped CV, March: rule P@50 0.512 · LR 0.636 · RF 0.608 (base 0.51) | Both models beat the rule *within* a month |
| One month forward (train Mar → score Apr): RF P@50 **0.14**, LR 0.54, base 0.56 | Neither model beat the base rate out of month. RF collapsed; LR merely matched it. **LR orders the queue as the less-bad option, and the queue is sold as a review order, not a forecast** |
| Walk-forward, fixed 23-client panel: LR 0.56–0.70 · RF 0.28–0.48 · rule 0.38 | Refit monthly; never reuse a stale model |
| Rule tie band: 24,462 pages share the top rule score (decline rate 50.7%) | The model's job is to order the tie band, not replace the rule |
| W6 error cases: top position (≤2) + CTR < 0.2% on 1.6k–17k impressions, clustered by client | That archetype gets a *diagnose* action, not a rewrite |

**Population reminder (survivorship, disclosed):** everything below describes pages with ≥14 covered days and ≥100 impressions in March 2026 *and* ≥14 covered days in April — 95,810 of 99,000 candidates. Pages whose tracking died are excluded, and those skew toward pages going quiet.

**Claim discipline:** the label is a 20% month-over-month impressions drop. This is cross-sectional, observational data. Nothing here says "refreshing page X will recover traffic". It says "these pages look worth a human's attention first, because…".

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### How the queue is ordered
1. **Primary key — out-of-fold LR probability of decline.** Probabilities are produced with `cross_val_predict` over the same five client-grouped folds as Week 5/6 (seed 42), so every page is scored by a model that never saw its client. That is the honest counterpart of the P@50 = 0.636 fold mean; an in-sample fit would look better and mean less.
2. **Tie-break — recent impressions.** Between two pages the model cannot separate, the one with more traffic at stake goes first.
3. **Hand-rule score is carried, not used to rank.** It stays visible so a reviewer can see how the queue relates to the Week 4 baseline.

### Archetype → action map (reason codes)
Each page gets exactly one archetype, checked in this order. Every rule is stated in feature terms so an editor can verify it by hand from the queue row.

| Order | Reason code | Rule (all on the 30-day feature window) | What it usually means | Recommended action | Automate? |
|---|---|---|---|---|---|
| 1 | `STRUCTURAL_LOW_CTR_TOP_POS` | position ≤ 2 **and** CTR < 0.20% | Brand/navigational query, SERP feature or sitelink absorbing clicks; W6's dominant *false-positive* pattern | **Diagnose only.** Check query mix and SERP layout in GSC. Do not rewrite. | Never |
| 2 | `HIGH_EXPOSURE_RISK` | impressions ≥ portfolio P75 **and** P(decline) ≥ 0.70 | Large footprint with a high decline score | Senior editor review within the sprint: factual refresh, structure, internal links | No — review first |
| 3 | `TOP_POS_CTR_EROSION` | position ≤ 10 **and** CTR < 1.0% | Visible but not chosen; title/snippet or intent mismatch | Title + meta rewrite, snippet/schema test | No — A/B only |
| 4 | `MATURE_CONTENT_DECAY` | age ≥ 180 days | Lifecycle staleness | Freshness pass: facts, dates, examples, dead links | No |
| 5 | `THIN_ACTIVITY_DRIFT` | active days < 25 of 30 | Intermittent impressions → crawl/index or demand gaps | Technical check before any editorial spend | No |
| 6 | `BROAD_EFFICIENCY_DECLINE` | everything else | Moderate signal, no single driver | Competitor gap review; lowest priority per hour | No |

Archetype 1 sits *first* on purpose: it is the pattern the model over-scores, and the correct response is to stop an editor from spending a refresh on it.

### What P@50 means here and what it does not
In client-grouped CV the top-50 of an LR-ranked queue held ~0.64 declining pages against a fold base rate of ~0.51 and a 0.507 tie-band rate. Read as decision-support: **roughly 13 more true declines per 100 reviewed than random tie-breaking, in the month the model was fit — and no measured lift one month later (W6: 0.54 vs 0.56).** The whole-frame queue precision printed below is out-of-fold but still within-month; it is a sanity check, not a result.

In [ ]:
# Setup: libraries, repo root, HF token, receipts from Weeks 4-6.
# Runs in Colab (clones the repo if needed) or locally from work/notebooks/.
%pip -q install duckdb huggingface_hub scikit-learn matplotlib

import os, json, getpass, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb, sklearn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, cross_val_predict

SEED = 42
np.random.seed(SEED)

# --- repo root: walk up from cwd looking for work/outputs; in bare Colab, clone the repo ---
def find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "work" / "outputs").is_dir() and (p / "skills").is_dir():
            return p
    return None

REPO = find_repo_root()
if REPO is None:
    target = Path("/content/FlyRank-ML-Internship")
    if not target.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Tessa-Saumu/FlyRank-ML-Internship.git", str(target)], check=True)
    REPO = target
OUT_DIR = REPO / "work" / "outputs"
FIG_DIR = REPO / "work" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

# --- receipts: every number quoted in prose comes from these files ---
w5 = json.loads((OUT_DIR / "model_metrics.json").read_text())
w6 = json.loads((OUT_DIR / "validation_audit_metrics.json").read_text())
w4 = json.loads((OUT_DIR / "baseline_metrics.json").read_text())

GCV = w6["grouped_cv"]["summary"]
TF = w6["time_forward"]
TIE = w6["rule_tie_band"]
EXPECTED_ROWS = w6["frames"]["march"]["rows"]

print("\n=== Receipts loaded ===")
print(f"Grouped-CV P@50 mean  rule {GCV['baseline']['p50_mean']:.3f} | LR {GCV['logistic_regression']['p50_mean']:.3f} | RF {GCV['random_forest']['p50_mean']:.3f}")
print(f"Time-forward P@50     rule {TF['metrics']['baseline']['p50']:.2f} | LR {TF['metrics']['logistic_regression']['p50']:.2f} | RF {TF['metrics']['random_forest']['p50']:.2f}  (April base {TF['test_base_rate']:.3f})")
print(f"Rule tie band         n={TIE['n']:,}  decline rate {TIE['decline_rate']:.3f}")

# --- HF token: env var -> Colab secret -> repo .env -> prompt. Never paste it in a cell. ---
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN and (REPO / ".env").exists():
    for line in (REPO / ".env").read_text().splitlines():
        if line.startswith("HF_TOKEN="):
            HF_TOKEN = line.split("=", 1)[1].strip()
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
con.execute("SET memory_limit='4GB'")
con.execute("SET threads=4")
REL = "hf://datasets/FlyRank/internship-warehouse"

In [ ]:
# Rebuild the Week-5/6 March development frame (features Mar 02-31, label Apr 01-30),
# then produce OUT-OF-FOLD probabilities with the same client-grouped folds and seed.
cutoff_date, recent_start = "2026-03-31", "2026-03-02"
label_start, label_end = "2026-04-01", "2026-04-30"
fact_march = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
fact_april = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
dim_content = f"read_parquet('{REL}/dim_content.parquet')"

df = con.sql(f'''
    WITH recent AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS recent30_impressions,
               SUM(gsc_clicks)      AS recent30_clicks,
               AVG(NULLIF(gsc_avg_position, 0)) AS recent30_avg_position,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS recent30_active_days,
               COUNT(DISTINCT report_date) AS recent30_days
        FROM {fact_march}
        WHERE report_date BETWEEN DATE '{recent_start}' AND DATE '{cutoff_date}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    future AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS future30_impressions,
               COUNT(DISTINCT report_date) AS future30_days
        FROM {fact_april}
        WHERE report_date BETWEEN DATE '{label_start}' AND DATE '{label_end}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT r.client_hash_id, r.content_hash_id,
           r.recent30_impressions,
           LN(1 + r.recent30_impressions) AS log_recent30_impressions,
           100.0 * r.recent30_clicks / NULLIF(r.recent30_impressions, 0) AS recent30_ctr_pct,
           r.recent30_avg_position,
           r.recent30_active_days,
           DATE_DIFF('day', c.content_created_date, DATE '{cutoff_date}') AS content_age_days,
           f.future30_impressions,
           CASE WHEN r.recent30_impressions >= 100
                 AND f.future30_impressions < 0.80 * r.recent30_impressions THEN 1 ELSE 0 END AS is_declining_next30
    FROM recent r
    INNER JOIN future f USING (client_hash_id, content_hash_id)
    LEFT JOIN (SELECT client_hash_id, content_hash_id, content_created_date FROM {dim_content}) c
           USING (client_hash_id, content_hash_id)
    WHERE r.recent30_days >= 14 AND f.future30_days >= 14 AND r.recent30_impressions >= 100
''').df()

assert not df.duplicated(["client_hash_id", "content_hash_id"]).any(), "Grain violation"
df = df.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
assert len(df) == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS:,} rows (W6 receipt), got {len(df):,}"

FEATURES = ["log_recent30_impressions", "recent30_ctr_pct", "recent30_avg_position",
            "recent30_active_days", "content_age_days"]
TARGET = "is_declining_next30"
assert "future30_impressions" not in FEATURES and "rule_score" not in FEATURES
X = df[FEATURES].fillna(df[FEATURES].median(numeric_only=True))
y = df[TARGET].values
groups = df["client_hash_id"].values
base_rate = float(y.mean())

# Same fold machinery as W5/W6. GroupKFold is deterministic given group order;
# the frame is sorted by client/content so folds reproduce.
gkf = GroupKFold(n_splits=5)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                            class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)

df["lr_proba_oof"] = cross_val_predict(lr, X, y, groups=groups, cv=gkf, method="predict_proba")[:, 1]
df["rf_proba_oof"] = cross_val_predict(rf, X, y, groups=groups, cv=gkf, method="predict_proba")[:, 1]

# Week-4 hand rule, carried for context only.
is_vis = (df["recent30_impressions"] >= 500).astype(int)
is_top10 = ((df["recent30_avg_position"] > 0) & (df["recent30_avg_position"] <= 10)).astype(int)
is_low_ctr = (df["recent30_ctr_pct"] < 1.0).astype(int)
is_stale = (df["content_age_days"] >= 91).astype(int)
df["rule_score"] = 0.40 * is_vis + 0.35 * (is_vis * is_top10 * is_low_ctr) + 0.25 * (is_vis * is_stale)

tie = df["rule_score"] == 1.0
print(f"Frame rows {len(df):,} | base rate {base_rate:.3f} | clients {df['client_hash_id'].nunique()}")
print(f"Rule tie band n={int(tie.sum()):,}, decline rate {df.loc[tie, TARGET].mean():.3f}  (W6 receipt: {TIE['n']:,}, {TIE['decline_rate']:.3f})")
assert int(tie.sum()) == TIE["n"], "Tie band does not match the W6 receipt"

# Out-of-fold P@K per fold (should sit near the W5/W6 grouped-CV means, not above them).
def p_at_k(frame, score, k=50):
    return frame.nlargest(k, score)[TARGET].mean()
fold_rows = []
for i, (_, te) in enumerate(gkf.split(X, y, groups), 1):
    f = df.iloc[te]
    fold_rows.append({"fold": i, "test_base": round(f[TARGET].mean(), 3),
                      "rule_p50": round(p_at_k(f, "rule_score"), 2),
                      "lr_p50": round(p_at_k(f, "lr_proba_oof"), 2),
                      "rf_p50": round(p_at_k(f, "rf_proba_oof"), 2)})
fold_tbl = pd.DataFrame(fold_rows)
print("\n=== Out-of-fold P@50 by client-grouped fold (rebuilt here) ===")
print(fold_tbl.to_string(index=False))
print(f"means: rule {fold_tbl.rule_p50.mean():.3f} | LR {fold_tbl.lr_p50.mean():.3f} | RF {fold_tbl.rf_p50.mean():.3f}")
print(f"W5 receipts: rule {GCV['baseline']['p50_mean']:.3f} | LR {GCV['logistic_regression']['p50_mean']:.3f} | RF {GCV['random_forest']['p50_mean']:.3f}")

In [ ]:
# Archetype -> action map, applied in priority order, then the ranked queue.
P75_IMP = float(df["recent30_impressions"].quantile(0.75))
SCORE = "lr_proba_oof"   # ranker chosen on W6 time-forward evidence (see header table)

ACTIONS = {
    "STRUCTURAL_LOW_CTR_TOP_POS": ("DIAGNOSE ONLY: check query mix, sitelinks and SERP features in GSC. Do not rewrite; "
                                   "this pattern was the dominant false positive in the W6 error analysis.", "never"),
    "HIGH_EXPOSURE_RISK":         ("Senior editorial review this sprint: verify facts, structure, internal links; "
                                   "large footprint at stake.", "no - review first"),
    "TOP_POS_CTR_EROSION":        ("Title/meta rewrite and snippet or schema test; measure CTR change against a holdout.", "no - A/B only"),
    "MATURE_CONTENT_DECAY":       ("Freshness pass: update facts, dates, examples, replace dead links.", "no"),
    "THIN_ACTIVITY_DRIFT":        ("Technical check first (crawl, index, canonical); no editorial spend until cleared.", "no"),
    "BROAD_EFFICIENCY_DECLINE":   ("Competitor gap review; lowest priority per editor-hour.", "no"),
}

def archetype(r):
    pos, ctr = r["recent30_avg_position"], r["recent30_ctr_pct"]
    if pd.notna(pos) and pos <= 2.0 and ctr < 0.20:
        return "STRUCTURAL_LOW_CTR_TOP_POS"
    if r["recent30_impressions"] >= P75_IMP and r[SCORE] >= 0.70:
        return "HIGH_EXPOSURE_RISK"
    if pd.notna(pos) and pos <= 10.0 and ctr < 1.0:
        return "TOP_POS_CTR_EROSION"
    if r["content_age_days"] >= 180:
        return "MATURE_CONTENT_DECAY"
    if r["recent30_active_days"] < 25:
        return "THIN_ACTIVITY_DRIFT"
    return "BROAD_EFFICIENCY_DECLINE"

df["reason_code"] = df.apply(archetype, axis=1)
df["recommended_action"] = df["reason_code"].map(lambda c: ACTIONS[c][0])
df["automatable"] = df["reason_code"].map(lambda c: ACTIONS[c][1])

# Human-readable reason string built from the actual feature values of the row.
def reason_text(r):
    parts = [f"P(decline)={r[SCORE]:.2f}"]
    parts.append(f"impr={int(r['recent30_impressions']):,}" + (" (>=P75)" if r["recent30_impressions"] >= P75_IMP else ""))
    if pd.notna(r["recent30_avg_position"]): parts.append(f"pos={r['recent30_avg_position']:.1f}")
    parts.append(f"ctr={r['recent30_ctr_pct']:.2f}%")
    parts.append(f"age={int(r['content_age_days'])}d" if pd.notna(r["content_age_days"]) else "age=NA")
    parts.append(f"active={int(r['recent30_active_days'])}/30")
    return "; ".join(parts)
df["reason_text"] = df.apply(reason_text, axis=1)

queue = df.sort_values([SCORE, "recent30_impressions"], ascending=[False, False]).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))
assert queue[SCORE].is_monotonic_decreasing

show = ["rank", "content_hash_id", "client_hash_id", SCORE, "rule_score", "recent30_impressions",
        "recent30_avg_position", "recent30_ctr_pct", "content_age_days", "reason_code"]
print("=== Top-20 of the review queue (ordered by out-of-fold LR probability) ===")
print(queue[show].head(20).round(3).to_string(index=False))

print("\n=== Archetype distribution: whole frame vs top-100 ===")
dist = pd.concat([df["reason_code"].value_counts().rename("frame_n"),
                  (df["reason_code"].value_counts(normalize=True) * 100).round(1).rename("frame_pct"),
                  df.groupby("reason_code")[TARGET].mean().round(3).rename("frame_decline_rate"),
                  queue.head(100)["reason_code"].value_counts().rename("top100_n")], axis=1).fillna(0)
print(dist.sort_values("frame_n", ascending=False).to_string())

top50 = queue.head(50)
print(f"\nSanity (out-of-fold, whole-frame top-50; NOT a headline metric): P@50 = {top50[TARGET].mean():.2f} "
      f"vs base {base_rate:.3f} vs tie band {TIE['decline_rate']:.3f}")
print(f"Distinct clients in top-50: {top50['client_hash_id'].nunique()} | largest client share: {top50['client_hash_id'].value_counts(normalize=True).iloc[0]:.0%}")

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use
- **Who:** a content lead or SEO strategist running a monthly refresh sprint across many client sites.
- **What:** a *review order*. The queue answers "which of the ~24k rule-tied pages should a human look at first this month?" It does not answer "what should be written" or "will this recover traffic".
- **When:** rebuilt on the first working day of each month from the previous month's features, with a model refit on that month. A queue older than its feature month is expired (Section 4 says why).
- **Scale:** the top 50–100 rows per cycle. Beyond that the score separation between neighbours is smaller than the fold-to-fold noise in Week 5's receipts (P@50 sd ≈ 0.19–0.23).

### The decay / refresh insight
Two things decay here, and the playbook has to respect both.

1. **Content decays with age — but only mildly, and not monotonically.** The cell below bins decline rate by content age. Age is a useful *archetype* input (it tells an editor what kind of refresh to do) but on its own it is a weak *ranker*.
2. **The ranking itself decays within weeks.** Week 6 measured it: a March-fit RF ranked April's top-50 at 0.14 precision against a 0.56 base rate; LR held at 0.54. On a fixed panel, adding more history did not fix it (LR peaked at two months of history, then fell). The base rate itself moved from 0.20 in February to 0.56 in April. Conclusion carried into this playbook: **the model is a one-cycle instrument. Refit monthly; never act on last month's queue.**

### Limits (read before trusting any row)
- **Observational, not causal.** The label is a ≥20% month-over-month impressions drop. Nothing here proves that a refresh changes that outcome.
- **Client- and time-dependent.** Grouped-fold P@50 ranged from 0.38 to 0.86 across client draws (W5); time-forward it fell to near base rate. Expect it to work best on large, stable clients and worst on new or small ones.
- **Label partly measures measurement continuity.** W6 showed decline rate of 96% for pages with 14–20 covered April days vs 40% for 30 days. Some "declines" are tracking gaps.
- **Survivorship.** 3.2% of candidates (tracking died in April) are excluded — exactly the pages most likely to be going quiet.
- **Ties and clusters.** W6 found 829 duplicate feature vectors in the frame and a 182-row single-client cluster that produced a perfect fold. Client-cluster effects are real; the concentration check below is a guard, not a fix.
- **Position ≤ 2 pages.** The model likes them; editors should not. See archetype 1.

In [ ]:
# Decay insight, part 1: decline rate by content age (observed, March->April frame).
age_bins = [0, 30, 90, 180, 365, 730, np.inf]
age_labels = ["<30d", "30-90d", "90-180d", "180-365d", "1-2y", ">2y"]
df["age_bucket"] = pd.cut(df["content_age_days"], bins=age_bins, labels=age_labels, right=False)
age_tbl = df.groupby("age_bucket", observed=True).agg(pages=(TARGET, "size"), decline_rate=(TARGET, "mean"),
                                                     median_impr=("recent30_impressions", "median")).reset_index()
age_tbl["pages_pct"] = (100 * age_tbl["pages"] / len(df)).round(1)
print("=== Observed decline rate by content age (n printed; buckets < 500 are noisy) ===")
print(age_tbl.round(3).to_string(index=False))
print(f"missing age: {df['content_age_days'].isna().sum():,} rows")

# Decay insight, part 2: model lift by evaluation design (all numbers from the W6 receipt).
wf = w6["walk_forward"]["origins"]
design_rows = [
    {"design": "grouped CV, March (mean of 5)", "base": round(w6['frames']['march']['base_rate'], 3),
     "rule": GCV["baseline"]["p50_mean"], "LR": GCV["logistic_regression"]["p50_mean"], "RF": GCV["random_forest"]["p50_mean"]},
    {"design": "time-forward: fit Mar -> score Apr", "base": round(TF["test_base_rate"], 3),
     "rule": TF["metrics"]["baseline"]["p50"], "LR": TF["metrics"]["logistic_regression"]["p50"], "RF": TF["metrics"]["random_forest"]["p50"]},
] + [{"design": f"walk-forward, history through {o['history_through']} -> Apr (fixed panel)", "base": round(o["test_base_rate"], 3),
      "rule": o["metrics"]["baseline"]["p50"], "LR": o["metrics"]["logistic_regression"]["p50"], "RF": o["metrics"]["random_forest"]["p50"]} for o in wf]
design_tbl = pd.DataFrame(design_rows)
print("\n=== P@50 by validation design (W6 receipts) ===")
print(design_tbl.to_string(index=False))

# Figure 1: two panels for the paper.
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].bar(age_tbl["age_bucket"].astype(str), age_tbl["decline_rate"], color="#4C72B0")
ax[0].axhline(base_rate, ls="--", c="grey", lw=1, label=f"frame base rate {base_rate:.2f}")
for i, (n, r) in enumerate(zip(age_tbl["pages"], age_tbl["decline_rate"])):
    ax[0].text(i, r + 0.01, f"n={n:,}", ha="center", fontsize=8)
ax[0].set_ylim(0, 1); ax[0].set_ylabel("observed decline rate (Apr vs Mar)"); ax[0].set_title("Content age vs decline rate (March 2026 frame)")
ax[0].legend(loc="upper left", fontsize=8)

xs = np.arange(len(design_tbl)); w = 0.25
for j, (m, c) in enumerate([("rule", "#999999"), ("LR", "#DD8452"), ("RF", "#55A868")]):
    ax[1].bar(xs + (j - 1) * w, design_tbl[m], w, label=m, color=c)
ax[1].scatter(xs, design_tbl["base"], marker="_", s=400, c="black", label="test base rate", zorder=3)
ax[1].set_xticks(xs); ax[1].set_xticklabels(["grouped CV\nMarch", "Mar->Apr", "wf thru Nov", "wf thru Dec", "wf thru Jan", "wf thru Feb"], fontsize=8)
ax[1].set_ylim(0, 1); ax[1].set_ylabel("precision@50"); ax[1].set_title("Ranking lift decays out of month (W6 receipts)")
ax[1].legend(fontsize=8, ncol=2)
plt.tight_layout()
fig1_path = FIG_DIR / "w07_fig1_decay_age_and_model.png"
plt.savefig(fig1_path, dpi=150); plt.close()
print("\nSaved", fig1_path)

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Review checklist (every queued page, before any edit)
The model sees five numbers. A reviewer must check the things it cannot see:

1. **Query mix.** Open the page's queries in GSC. If impressions are dominated by brand or navigational terms, the page is not "decaying" — mark `NOT_ACTIONABLE`.
2. **SERP layout.** Did an AI Overview, featured snippet, video carousel or ad block appear above the result? CTR loss from layout is not a content problem.
3. **Technical state.** Indexing status, canonical, redirects, Core Web Vitals, recent template changes. Fix technical before editorial.
4. **Seasonality.** Compare to the same window last year where history exists; a seasonal dip needs no action.
5. **Business value.** Does the page matter (conversion, compliance, brand)? Traffic at risk is not value at risk.
6. **YMYL / regulated?** If yes, route to the subject-matter owner regardless of score.

The reviewer records one of `ACT / DEFER / NOT_ACTIONABLE / ESCALATE` next to each row. Those decisions are the labels a future version of this model should learn from — they are worth more than another month of impressions.

### Automated safety gates
The cell below flags rows that need a senior look *before* ordinary review. Gates are heuristics on the same five features, so they can only catch what the features express; they narrow the human job, they do not replace it.

### The no-go list — never automated, whatever the score
- **No automated deletions, redirects, noindex, or consolidation.** Irreversible, destroys link equity, and the model has no idea why a page exists.
- **No unreviewed generative rewrites pushed live.** Factual and brand risk; the label cannot tell good content from bad.
- **No automated edits to YMYL, legal, medical, financial, or safety pages.** Owner review is mandatory.
- **No bulk template or navigation changes** triggered by page-level scores; those are site decisions.
- **No per-client SLAs or reporting built on this score.** Fold-to-fold P@50 sd ≈ 0.2 — it is not stable enough to promise a number to a client.
- **No acting on `STRUCTURAL_LOW_CTR_TOP_POS` rows** beyond diagnosis. The model over-scores them; a rewrite is the wrong tool.
- **No use of a queue older than its feature month.**

In [ ]:
# Safety gates on the top-100, then the reviewer worksheet columns.
def safety_gates(r):
    flags = []
    if pd.notna(r["content_age_days"]) and r["content_age_days"] < 30:
        flags.append("NEW_PAGE: <30d old, ranking not settled")
    if pd.notna(r["recent30_avg_position"]) and r["recent30_avg_position"] <= 2.0:
        flags.append("TOP2_POSITION: protect, do not restructure")
    if pd.notna(r["recent30_avg_position"]) and r["recent30_avg_position"] <= 5.0 and r["recent30_ctr_pct"] < 0.20:
        flags.append("SERP_LAYOUT_SUSPECT: near-zero CTR at high position")
    if r["recent30_active_days"] < 14:
        flags.append("THIN_COVERAGE: <14 active days")
    return "; ".join(flags) if flags else "CLEARED_FOR_REVIEW"

queue["safety_gate"] = queue.apply(safety_gates, axis=1)
queue["review_decision"] = ""          # ACT / DEFER / NOT_ACTIONABLE / ESCALATE - filled by a human
queue["reviewer_note"] = ""

top100 = queue.head(100)
gate_status = np.where(top100["safety_gate"] == "CLEARED_FOR_REVIEW", "CLEARED", "SENIOR_LOOK_FIRST")
print("=== Safety gates on the top-100 ===")
print(pd.Series(gate_status).value_counts().to_string())
print("\nFlag breakdown:")
print(top100.loc[top100["safety_gate"] != "CLEARED_FOR_REVIEW", "safety_gate"].str.split("; ").explode().str.split(":").str[0].value_counts().to_string())

# Client concentration guard on the top-100 (W6 measured 22% as the largest client's frame share).
cc = top100["client_hash_id"].value_counts(normalize=True)
print(f"\nTop-100 spans {cc.size} clients; largest share {cc.iloc[0]:.0%} (frame share of largest client: {w6['frames']['march']['largest_client_share']:.0%})")
if cc.iloc[0] > 0.35:
    print("WARNING: one client holds >35% of the queue - cap per-client rows before handing to editors.")

print("\n=== Example rows an editor sees (rank, archetype, reason, gate) ===")
print(top100[["rank", "reason_code", "reason_text", "safety_gate"]].head(8).to_string(index=False))

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

This is a notebook run monthly by one person, not a service. Monitoring therefore means **a short table computed at each rebuild**, with thresholds set from the W5/W6 receipts:

| # | Trigger | Threshold | Why this number | Response |
|---|---|---|---|---|
| 1 | Base-rate drift | frame decline rate moves > 10 pp from the fit month | W6 saw 0.20 → 0.56 across Feb–Apr; a 10 pp move already changes what "P@50" means | Refit before ranking; re-read every threshold |
| 2 | Out-of-time lift | last month's queue P@50 (now labelled) < last month's base rate + 0.05 | W6 time-forward RF: 0.14 vs 0.56 — the failure mode this catches | Do not ship the queue; fall back to the hand rule + reviewer judgment |
| 3 | Client concentration | largest client > 35% of top-100 | W6: largest client is 22% of the frame; above 35% the queue is one client's problem | Cap per-client rows |
| 4 | Panel churn | > 20% of frame rows from clients absent in the fit month | grouped folds swing 0.38–0.86 across client draws | Refit; flag new clients' rows for extra review |
| 5 | Coverage floor | any row with < 14 covered days in the feature window | data contract from W3; label-continuity confound measured in W6 | Halt; fix the frame query |
| 6 | Reviewer disagreement | > 50% of top-50 marked `NOT_ACTIONABLE` | human labels beat the proxy label | Revisit archetype rules and the label definition |

**Retrain cadence:** every cycle, unconditionally. Triggers 1–2 decide whether the *fresh* fit is allowed to rank at all. There is no "model in production" to roll back; there is a notebook and a CSV.

**Honest status at this first cycle:** on the W6 evidence, trigger 2 fires for *both* rankers — one month forward, LR matched the base rate and RF fell far below it. This playbook therefore does not claim the queue predicts next month. It claims a defensible *review order* (grouped-CV lift within the month, archetype rules that stop the known false positive, exposure tie-break) plus the discipline to measure last month's queue before trusting this month's. If the next cycle's labelled P@50 also fails to clear base + 0.05, the model should be dropped from the workflow and the archetype rules kept.

**Cost / value framing (illustrative, not measured).** Editor time is the scarce resource. Per archetype, assume a rough hours-per-action (stated in the cell) and value-at-stake ≈ recent impressions × P(decline). Rank *within* the shortlist by value per hour, and drop archetype 1 rows from the effort budget entirely (diagnosis is minutes, not a refresh). These are planning numbers a content lead should replace with their own rates; the point is to show that the cheapest, highest-exposure actions are not always the highest-probability rows.

In [ ]:
# Monitoring table computed at this rebuild. Trigger 2 uses the W6 out-of-time numbers as
# "last month's queue" since this is the first cycle; in a real rebuild it is last month's CSV, labelled.
FIT_BASE = w6["frames"]["march"]["base_rate"]
checks = []
def add(name, value, threshold, ok, response):
    checks.append({"trigger": name, "value": value, "threshold": threshold, "status": "PASS" if ok else "TRIGGERED", "response": response})

drift = abs(base_rate - FIT_BASE)
add("1 base-rate drift", f"{drift:.3f}", "<= 0.10", drift <= 0.10, "refit before ranking")
for m, key in [("LR", "logistic_regression"), ("RF", "random_forest")]:
    p50, b = TF["metrics"][key]["p50"], TF["test_base_rate"]
    add(f"2 out-of-time lift ({m}, W6 Mar->Apr)", f"P@50 {p50:.2f} vs base {b:.2f}", "P@50 >= base + 0.05", p50 >= b + 0.05,
        "do not ship queue; fall back to rule + review")
add("3 client concentration (top-100)", f"{cc.iloc[0]:.0%}", "<= 35%", cc.iloc[0] <= 0.35, "cap per-client rows")
add("4 panel churn", "0% (single frame this cycle)", "<= 20%", True, "refit; flag new clients")
thin = float((df["recent30_active_days"] < 14).mean())
add("5 coverage floor", f"{thin:.1%} rows <14 covered days", "0%", thin == 0.0, "halt; fix frame query")
add("6 reviewer disagreement", "n/a - no reviews yet", "<= 50% NOT_ACTIONABLE", True, "revisit archetypes + label")

monitor = pd.DataFrame(checks)
print("=== Monitoring at this rebuild ===")
print(monitor.to_string(index=False))
triggered = monitor.loc[monitor["status"] == "TRIGGERED", "trigger"].tolist()
print("\nTriggered:", triggered if triggered else "none")
lr_fires = any("LR" in t for t in triggered); rf_fires = any("RF" in t for t in triggered)
print("Reading:", "trigger 2 fires for BOTH rankers on W6 evidence: neither cleared base + 0.05 one month forward. "
      "LR is the less-bad ranker (0.54 vs RF 0.14) and orders the queue, but the honest status of this cycle is "
      "'review order, not a forecast' - the rule and reviewer judgment carry the decision." if (lr_fires and rf_fires)
      else f"LR triggered={lr_fires}, RF triggered={rf_fires}.")

# --- Cost / value (illustrative planning numbers; replace with your own) ---
HOURS = {"STRUCTURAL_LOW_CTR_TOP_POS": 0.25, "HIGH_EXPOSURE_RISK": 6.0, "TOP_POS_CTR_EROSION": 1.5,
         "MATURE_CONTENT_DECAY": 3.0, "THIN_ACTIVITY_DRIFT": 1.0, "BROAD_EFFICIENCY_DECLINE": 4.0}
queue["est_hours"] = queue["reason_code"].map(HOURS)
queue["impr_at_risk"] = queue["recent30_impressions"] * queue[SCORE]          # expected impressions exposed to a >=20% drop
queue["value_per_hour"] = queue["impr_at_risk"] / queue["est_hours"]
queue.loc[queue["reason_code"] == "STRUCTURAL_LOW_CTR_TOP_POS", "value_per_hour"] = np.nan   # diagnosis, not refresh

shortlist = queue.head(100)
budget = shortlist.groupby("reason_code").agg(rows=("rank", "size"), hours=("est_hours", "sum"),
                                              impr_at_risk=("impr_at_risk", "sum")).sort_values("hours", ascending=False)
budget["impr_at_risk"] = budget["impr_at_risk"].round(0)
print("\n=== Effort budget for the top-100 by archetype (illustrative hours) ===")
print(budget.to_string())
print(f"Total: {shortlist['est_hours'].sum():.0f} editor-hours for 100 rows; "
      f"{(shortlist['reason_code']=='STRUCTURAL_LOW_CTR_TOP_POS').sum()} rows are diagnosis-only.")

print("\n=== Same top-100, re-ordered by value per editor-hour (top 10) ===")
print(shortlist.sort_values("value_per_hour", ascending=False)[["rank", "reason_code", SCORE, "recent30_impressions", "est_hours", "value_per_hour"]]
      .head(10).round(2).to_string(index=False))

# Figure 2: archetype mix and probability-vs-exposure of the top-100.
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
mix = shortlist["reason_code"].value_counts()
ax[0].barh(mix.index, mix.values, color="#4C72B0"); ax[0].invert_yaxis()
ax[0].set_title("Archetype mix of the top-100 review queue"); ax[0].set_xlabel("rows")
colors = shortlist["reason_code"].astype("category").cat.codes
sc = ax[1].scatter(shortlist[SCORE], shortlist["recent30_impressions"], c=colors, cmap="tab10", s=28, alpha=0.8)
ax[1].set_yscale("log"); ax[1].set_xlabel("out-of-fold P(decline), LR"); ax[1].set_ylabel("30-day impressions (log)")
ax[1].set_title("Top-100: probability vs exposure")
handles = [plt.Line2D([], [], marker="o", ls="", color=plt.cm.tab10(i / 10), label=c)
           for i, c in enumerate(shortlist["reason_code"].astype("category").cat.categories)]
ax[1].legend(handles=handles, fontsize=7, loc="lower left")
plt.tight_layout()
fig2_path = FIG_DIR / "w07_fig2_queue_archetypes.png"
plt.savefig(fig2_path, dpi=150); plt.close()
print("\nSaved", fig2_path)

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

| File | Committed? | What the paper uses it for |
|---|---|---|
| `work/outputs/action_playbook_queue.csv` | **No** (gitignored by the leak-guard; regenerated by this notebook) | The ranked top-100 with reason codes, actions, gates, effort estimate and empty reviewer columns |
| `work/outputs/action_playbook_summary.json` | **Yes** | Receipt: frame size, base rate, tie band, out-of-fold fold table, archetype distribution, gate counts, monitoring statuses, effort assumptions, library versions |
| `work/figures/w07_fig1_decay_age_and_model.png` | **Yes** | Decay/refresh figure for the recommendations section |
| `work/figures/w07_fig2_queue_archetypes.png` | **Yes** | Queue composition figure |

Every number quoted in the prose above either comes from a W4–W6 receipt or is written into this summary JSON.

In [ ]:
# Exports.
export_cols = ["rank", "content_hash_id", "client_hash_id", SCORE, "rf_proba_oof", "rule_score",
               "recent30_impressions", "recent30_avg_position", "recent30_ctr_pct", "content_age_days",
               "recent30_active_days", "reason_code", "reason_text", "recommended_action", "automatable",
               "safety_gate", "est_hours", "impr_at_risk", "value_per_hour", "review_decision", "reviewer_note"]
queue_out = queue.head(100)[export_cols].round(4)
csv_path = OUT_DIR / "action_playbook_queue.csv"
queue_out.to_csv(csv_path, index=False)
print(f"Wrote {len(queue_out)} rows -> {csv_path}  (gitignored; regenerate by running this notebook)")

summary = {
    "notebook": "w07_action_playbook",
    "seed": SEED,
    "ranker": "logistic_regression (StandardScaler + class_weight=balanced), out-of-fold via GroupKFold(5) on client_hash_id",
    "ranker_choice_evidence": {"grouped_cv_p50": {k: GCV[k]["p50_mean"] for k in GCV},
                               "time_forward_p50": {k: TF["metrics"][k]["p50"] for k in TF["metrics"]},
                               "time_forward_base_rate": TF["test_base_rate"]},
    "frame": {"feature_window": [recent_start, cutoff_date], "label_window": [label_start, label_end],
              "rows": int(len(df)), "clients": int(df["client_hash_id"].nunique()), "base_rate": base_rate,
              "largest_client_share": float(df["client_hash_id"].value_counts(normalize=True).iloc[0])},
    "rule_tie_band": {"n": int(tie.sum()), "decline_rate": float(df.loc[tie, TARGET].mean())},
    "oof_fold_p50": fold_tbl.to_dict(orient="records"),
    "oof_p50_means": {"rule": float(fold_tbl.rule_p50.mean()), "lr": float(fold_tbl.lr_p50.mean()), "rf": float(fold_tbl.rf_p50.mean())},
    "whole_frame_top50_sanity": {"p50": float(top50[TARGET].mean()), "note": "out-of-fold but whole-frame; sanity check, not a headline"},
    "queue_size": int(len(queue_out)),
    "archetype_rules_order": list(ACTIONS.keys()),
    "p75_impressions_threshold": P75_IMP,
    "archetype_distribution_frame": {k: int(v) for k, v in df["reason_code"].value_counts().items()},
    "archetype_decline_rate_frame": {k: float(v) for k, v in df.groupby("reason_code")[TARGET].mean().items()},
    "archetype_distribution_top100": {k: int(v) for k, v in queue.head(100)["reason_code"].value_counts().items()},
    "safety_gates_top100": {"cleared": int((gate_status == "CLEARED").sum()), "senior_look_first": int((gate_status != "CLEARED").sum())},
    "top100_clients": int(cc.size), "top100_largest_client_share": float(cc.iloc[0]),
    "age_bucket_decline_rate": age_tbl.assign(age_bucket=age_tbl["age_bucket"].astype(str)).to_dict(orient="records"),
    "monitoring": monitor.drop(columns=["response"]).to_dict(orient="records"),
    "monitoring_triggered": triggered,
    "effort_hours_assumed": HOURS,
    "top100_total_est_hours": float(shortlist["est_hours"].sum()),
    "figures": [str(fig1_path.relative_to(REPO)), str(fig2_path.relative_to(REPO))],
    "library_versions": {"pandas": pd.__version__, "numpy": np.__version__, "scikit-learn": sklearn.__version__, "duckdb": duckdb.__version__},
}
json_path = OUT_DIR / "action_playbook_summary.json"
json_path.write_text(json.dumps(summary, indent=2, default=float))
print(f"Wrote receipt -> {json_path}")
print("\nSummary head:", json.dumps({k: summary[k] for k in ["frame", "oof_p50_means", "monitoring_triggered", "safety_gates_top100"]}, indent=1, default=float))

# Colab convenience: bundle the committed artefacts so they can be added to the repo.
try:
    from google.colab import files  # noqa
    import shutil
    bundle = Path("/content/w07_artifacts")
    bundle.mkdir(exist_ok=True)
    for p in [json_path, fig1_path, fig2_path]:
        shutil.copy(p, bundle / p.name)
    shutil.make_archive("/content/w07_artifacts", "zip", bundle)
    files.download("/content/w07_artifacts.zip")
    print("Downloaded w07_artifacts.zip - commit its contents to work/outputs/ and work/figures/.")
except Exception:
    print("Not in Colab (or download skipped) - artefacts are already in the repo working tree.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Skeptic pass (do this after the real run, before ticking the boxes):**
- Read the top-20 table. Are the first rows `STRUCTURAL_LOW_CTR_TOP_POS`? If so, that is the model doing what W6 said it would — the archetype label is what protects the editor. Say so in the paper.
- Compare the rebuilt out-of-fold fold means to the W5 receipts. They should be within a few hundredths. If they are not, the folds or the model spec drifted — fix before quoting anything.
- Does any sentence above claim a *result* about refresh outcomes? There is no experiment here. Rewrite it as decision-support.